# kaiming-uniform-sf-init — worked example 1: SF init for various fan-in sizes — verify scale factor shrinks with width

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `kaiming-uniform-sf-init`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The scale factor `sf = 1/sqrt(fan_in)` controls how wide the uniform distribution is. A larger `fan_in` gives a smaller `sf`, meaning each weight starts closer to zero. The uniform sample is drawn on `(-sf, +sf)` using `t.rand * 2 - 1) * sf` to shift from `[0,1)` to the symmetric range.

## Worked solution

Step 1: Compute `sf = in_features ** -0.5` (equivalent to `1/sqrt(in_features)`).

Step 2: Draw uniform samples on `[0, 1)` with `t.rand(in_features, out_features, generator=g)`.

Step 3: Transform to `(-sf, +sf)` via `(raw * 2 - 1) * sf`. Multiplying by 2 stretches `[0,1)` to `[0,2)`, subtracting 1 gives `[-1,1)`, multiplying by `sf` scales to `(-sf, sf)`.

Step 4: Verify the resulting tensor's shape is `(in_features, out_features)` and all values fall strictly within `[-sf, sf]`.

In [ ]:
import torch as t

def kaiming_sf_init(in_features: int, out_features: int, generator: t.Generator) -> t.Tensor:
    sf = in_features ** -0.5
    raw = t.rand(in_features, out_features, generator=generator)
    return (raw * 2 - 1) * sf

# Demonstrate across different fan-in sizes
widths = [4, 32, 256]
g = t.Generator()

print(f'{'fan_in':>8}  {'sf':>8}  {'emp_min':>9}  {'emp_max':>9}')
for fan_in in widths:
    g.manual_seed(0)
    w = kaiming_sf_init(fan_in, 64, g)
    sf = fan_in ** -0.5
    print(f'{fan_in:>8}  {sf:>8.4f}  {w.min().item():>9.4f}  {w.max().item():>9.4f}')
    assert w.shape == (fan_in, 64)
    assert w.min().item() >= -sf - 1e-6
    assert w.max().item() <=  sf + 1e-6

print('All shapes and bounds verified.')